# Day 2, Notebook 1: package the logic, survive bad data

Yesterday your answers lived in cells. Today they become things you can call again, on records you have not seen yet.

Run this notebook top to bottom. Every failure in it is deliberate, and each one is captured so the notebook keeps running. Read the error text before you read the fix.

Position in the day:

`[inline cell] > [packaged decision] > [read the failure] > [log the rejection] > [cross the boundary]`

## Setup

One cell, at the top, so this notebook runs cold in a fresh Codespace.

In [ ]:
import csv
import traceback

RECORDS_CSV = "C2_W01_D02_data_records_STUDENT.csv"

with open(RECORDS_CSV) as f:
    records = list(csv.DictReader(f))

def show_failure(fn):
    """Run something that is meant to fail and print its real traceback."""
    try:
        fn()
    except Exception:
        print(traceback.format_exc())

print(f"{len(records)} records loaded")
print(records[0])

## Section 1: the cell you have already written three times

This is Monday's code, character for character. It worked yesterday.

Today it is pointed at today's file, and today's file came from somewhere you do not control.

In [ ]:
total = 0
for r in records:
    if r["outcome"] == "accepted":
        total = total + int(r["amount"])
print(total)

The code did not change. The data did. That is most of this job.

We come back to the error itself in section 4. For now, notice the shape of the problem before the error: the rule lives in a cell. To use it on a second dataset you copy the cell. To change the rule you edit every copy.

A rule that lives in three cells is three rules, and the copy you forget is the one that ships.

## Section 2: the same rule with a name

Section 1 plus one new element: the rule gets a name and one home.

Mental model:

```
records  ->  [ accepted_total ]  ->  a number
             the promise: give me records, I give you a number
```

The function can see only what you hand it. That is all of scope you need today.

In [ ]:
def accepted_total(some_records):
    total = 0
    for r in some_records:
        if r["outcome"] == "accepted":
            total = total + int(r["amount"])
    return total

# It fails the same way the cell did, for the same reason. Captured so we can keep reading.
show_failure(lambda: accepted_total(records))

The failure moved with the rule, which is the point. Naming code does not fix code. It gives the fix one place to live.

Work on the eight records that do convert, so the shape of the function is visible before we handle the defects.

In [ ]:
first_eight = records[:8]
print(accepted_total(first_eight))

### Milestone: where this shows up in production

Every data pipeline you will ever read is a chain of named functions like `accepted_total`, one per decision, called in order. The name is what lets a colleague review your logic without reading it. When an incident report says "the rule changed in one place and three callers were not updated", that report is about functions.

### Interview question this milestone just made answerable

"What does a function actually buy you over a copied block of code?"

Your answer has three parts now: one home for the rule, a name you can say out loud, and reuse on inputs you have not seen.

## Section 3: return against print

Section 2 plus one new element: what the function hands back.

`print` shows a human. `return` hands a value to the next line of code. Confusing the two is the most common first-week bug in any language.

In [ ]:
def fix_printing(record):
    print(record["id"])

def fix_returning(record):
    return record["id"]

printed = fix_printing(records[0])
returned = fix_returning(records[0])

print("printed holds:", repr(printed))
print("returned holds:", repr(returned))

### The deliberate failure

`fix_printing` showed you something and handed back nothing. Nothing is `None`. Now use it as though it were a value.

In [ ]:
show_failure(lambda: fix_printing(records[0])["id"])

Read the last line first:

```
TypeError: 'NoneType' object is not subscriptable
```

The type is `TypeError`. The value it was holding was `None`. `None` came from a function that printed instead of returning.

### The fix

In [ ]:
record_id = fix_returning(records[0])
print("the caller can use this:", record_id)
print("and pass it straight on:", record_id.startswith("10"))

Rule for the rest of the programme: if the caller needs the answer, the function returns it. Print is for you, standing at the screen.

## Section 4: catch the failure you expected

Section 3 plus one new element: surviving a bad value on purpose.

A traceback is the interpreter telling you where it stopped and what it was holding. Read it bottom-up:

```
last line      what went wrong        the exception type and the value
line above     where                  the line that is yours
everything else how you got there     the call chain
```

In [ ]:
show_failure(lambda: int("twelve"))

Three questions, every time:

1. What is the exception type? `ValueError`.
2. Which line is mine? The one calling `int`.
3. What value was it holding? `'twelve'`.

The third question is the one people skip, and it is the one that names the record.

### Catching it narrowly

In [ ]:
def to_amount(raw):
    """Convert an amount, or raise ValueError with the interpreter's own wording."""
    return int(raw)

converted = 0
failed = 0
for r in records:
    try:
        to_amount(r["amount"])
        converted += 1
    except ValueError:
        failed += 1

print(f"converted {converted}, failed {failed}")

### The deliberate failure that does not look like one

This next cell runs. It prints a number. The number is wrong and nothing on screen says so.

In [ ]:
total = 0
for r in records:
    try:
        total += int(r["amount"])
    except:
        pass
print(f"Processed {len(records)} records. Total: {total}")

Now the honest version of exactly the same work.

In [ ]:
total = 0
kept = 0
for r in records:
    try:
        total += int(r["amount"])
    except ValueError:
        continue
    kept += 1
print(f"Clean {kept}, rejected {len(records) - kept}, total {total}")

Both cells printed `230380`. The first one claimed thirty records went into it. Two did not.

That is the whole argument. A crash costs you an hour of your own time. A plausible wrong number costs you a quarter, because nobody goes looking for a number that looks fine.

The bare `except` also swallows the failures you did not think of: a typo in a key name, an interrupted keyboard, a bug three functions down.

### Milestone: where this shows up in production

Knight Capital, 1 August 2012, lost about USD 440 million in 45 minutes. A deployment reused an old flag and the system did not fail loudly. It kept trading, at speed, on the wrong rule.

Public Health England, October 2020, dropped 15,841 COVID cases from reporting when a CSV was converted into an old Excel format with a hard row limit. Nothing crashed. The rows past the limit were silently discarded.

### Interview question this milestone just made answerable

"Why is a bare `except` worse than letting the code crash?"

Answer with the two lines above as evidence: both printed the same total, and only one of them told the truth about how many records it covered.

## Section 5: the rejection is a deliverable

Section 4 plus one new element: the bad record goes somewhere, with a reason.

You have three choices when a record will not convert. Fix it silently, drop it silently, or set it aside with a reason. Only the third survives a question from someone who was not in the room.

In [ ]:
def clean_records(rows):
    """Return the rows that converted and the rows that did not, each rejection with its reason."""
    clean = []
    rejects = []
    for r in rows:
        try:
            amount = int(r["amount"])
        except ValueError as e:
            rejects.append({"id": r["id"], "reason": str(e)})
            continue
        keeper = dict(r)
        keeper["amount"] = amount
        clean.append(keeper)
    return clean, rejects

clean, rejects = clean_records(records)
print(f"input {len(records)}, clean {len(clean)}, rejected {len(rejects)}")
for row in rejects:
    print(row)

`str(e)` carries the interpreter's own wording. You never have to invent an error message, and the wording stays consistent across everyone's code.

### The reconciliation

This is the check that catches the silent loss, and you will use it every day from tomorrow.

In [ ]:
assert len(clean) + len(rejects) == len(records), "records went missing"
print(f"{len(records)} in = {len(clean)} clean + {len(rejects)} rejected")

When that sum does not hold, something disappeared and you do not yet know what. The assertion is cheap. The missing rows are not.

### Milestone: interview question

"Your cleaning run reported zero rejects on a file you know is dirty. What do you check?"

Three checks, in order: is the rejects list actually being appended to, is the `except` narrow enough to be reached, and does input equal clean plus rejected.

## Crux

A function is a decision you can call again. A named exception is a failure you chose to survive. A rejects log is the difference between a number and a number you can defend.

## What tomorrow does with this

`clean_records` gets pointed at the whole dataset tomorrow, without one edit, and the question becomes how many usable records the dataset actually has.

Carry these two functions forward. You will be asked for them by name.